# ETL — Protein Structural Quality Dataset (CASP)

## Objectives

- Load the CASP protein tertiary structure dataset.
- Inspect dataset dimensions, data types, and descriptive statistics.
- Identify missing values, duplicate records, and potential data quality issues.
- Investigate duplicate observations before deciding how to handle them.
- Export a cleaned dataset for exploratory analysis and machine learning.

## Inputs

- data/raw/CASP.csv

## Outputs

- data/processed/casp_cleaned.csv
- Summary of data quality checks and cleaning decisions.

## Additional Comments

The CASP dataset contains physicochemical and structural
descriptors of predicted protein models.

The target variable is RMSD, a continuous measure of
structural deviation.

Exact duplicate records will be investigated because they
may affect model evaluation and introduce leakage if
identical records occur in both training and test sets.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [7]:

import os
from pathlib import Path

current_dir = Path.cwd()
print("Current directory:", current_dir)

Current directory: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [5]:
if Path.cwd().name == "jupyter_notebooks":
    os.chdir(Path.cwd().parent)

print("Working directory set to:", Path.cwd())

Working directory set to: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


Confirm the new current directory

In [6]:
print("Confirmed project directory:", Path.cwd())

Confirmed project directory: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


# Section 1 — Data Loading and Initial Inspection

In [8]:
import pandas as pd
import numpy as np

# Load the original CASP dataset
df = pd.read_csv("data/raw/CASP.csv")

# Display dataset dimensions
print("Dataset shape:", df.shape)

# Display the first five observations
display(df.head())

Dataset shape: (45730, 10)


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147


In [9]:
# Inspect dataset structure and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45730 entries, 0 to 45729
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RMSD    45730 non-null  float64
 1   F1      45730 non-null  float64
 2   F2      45730 non-null  float64
 3   F3      45730 non-null  float64
 4   F4      45730 non-null  float64
 5   F5      45730 non-null  float64
 6   F6      45730 non-null  float64
 7   F7      45730 non-null  float64
 8   F8      45730 non-null  int64  
 9   F9      45730 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 3.5 MB


---

### Initial Dataset Structure — Findings

The CASP dataset contains 45,730 observations and 10 numerical
variables, including the target variable RMSD and nine predictors
(F1–F9).

All columns contain 45,730 non-null observations, indicating
that no missing values were detected during the initial inspection.

Nine variables have the float64 data type, while F8 has the
int64 data type.

The dataset requires approximately 3.5 MB of memory and is
suitable for local analysis.

Further data quality checks will investigate duplicate records,
invalid values, and numerical distributions before cleaning.

In [10]:
# Generate descriptive statistics for all numerical variables
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RMSD,45730.0,7.748528e+00,6.118312,0.0000,2.305000,5.030000e+00,1.337900e+01,2.099900e+01
F1,45730.0,9.871597e+03,4058.138034,2392.0500,6936.680000,8.898805e+03,1.212615e+04,4.003490e+04
F2,45730.0,3.017367e+03,1464.324663,403.5000,1979.045000,2.668155e+03,3.786410e+03,1.531200e+04
F3,45730.0,3.023919e-01,0.062886,0.0925,0.258740,3.001500e-01,3.428900e-01,5.776900e-01
F4,45730.0,1.034924e+02,55.424985,10.3101,63.563900,8.774080e+01,1.336467e+02,3.693170e+02
F5,45730.0,1.368299e+06,564036.688407,319490.2166,953591.221150,1.237219e+06,1.690920e+06,5.472011e+06
F6,45730.0,1.456381e+02,69.999230,31.9704,94.757500,1.261760e+02,1.814685e+02,5.984080e+02
F7,45730.0,3.989756e+03,1993.574575,0.0000,3165.322500,3.840170e+03,4.644193e+03,1.059482e+05
F8,45730.0,6.997507e+01,56.493443,0.0000,31.000000,5.400000e+01,9.100000e+01,3.500000e+02
F9,45730.0,3.452366e+01,5.979755,15.2280,30.424725,3.529930e+01,3.887080e+01,5.530090e+01


### Descriptive Statistics — Initial Findings

The CASP dataset contains 45,730 observations, with RMSD as
the continuous target variable and nine numerical predictors.

**Target variable (RMSD):**
- Mean: 7.75 Å
- Median: 5.03 Å
- Standard deviation: 6.12 Å
- Range: 0.00–21.00 Å

The mean RMSD exceeds the median, suggesting a positively
skewed distribution. This will be investigated during EDA.

**Predictor variables:**
- The predictors exhibit substantially different numerical scales.
- F5 contains values in the millions, whereas F3 contains
  values below 1.
- F1 and F7 show large differences between their median
  and maximum values, indicating potential upper-tail outliers.
- F8 contains integer-valued observations.

**Implications for preprocessing:**

Feature scaling will be considered within the machine learning
pipelines, particularly for models sensitive to feature scale.

Potential outliers will be investigated before any removal
decisions are made. Extreme values are not necessarily errors
and may represent legitimate protein structural characteristics.

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
